# Validation Test - Droplet spreading under gravity

This notebook and the corresponding evaluation notebook (DropletSpreadingUnderGravity_Evaluation.ipynb) are part of published results (cf. 5.2) found in *The extended discontinuous Galerkin method adapted for moving contact line problems via the generalized Navier boundary condition* (https://doi.org/10.1002/fld.5016).

In [ ]:
#r "BoSSSpad.dll"
//#r "..\..\..\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [2]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client LocalPC @C:\BoSSS-localJobs\binaries
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries"


In [3]:
string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
userName

FDY\smuda

In [4]:
//var myBatch = ExecutionQueues[1];
var myBatch = (userName.Equals(@"FDY\jenkinsci")) ? GetDefaultQueue() : ExecutionQueues[1];
myBatch

RuntimeLocation,win\amd64
AdditionalEnvironmentVars,[ ]
DeploymentBaseDirectory,\\dc3\userspace\smuda\hpccluster\binaries
DeployRuntime,True
Name,FDY-WindowsHPC
DotnetRuntime,dotnet
Username,FDY\smuda
NumOfServiceCoresPerMPIprocess,1
ServerName,DC3
ComputeNodes,<null>
DefaultJobPriority,Normal


In [5]:
wmg.Init("CLpaper_DropletSpreadingUnderGravity", myBatch);

Project name is set to 'CLpaper_DropletSpreadingUnderGravity'.
Opening existing database '\\dc3\userspace\smuda\hpccluster\CLpaper_DropletSpreadingUnderGravity'.


In [6]:
wmg.SetNameBasedSessionJobControlCorrelation();

In [7]:
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [ ]:
static var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions

In [ ]:
// foreach (var s in sessions) {
//     if(s.Name.Contains("StaticDropletOnSlipWall_Eo5") && !s.Name.Contains("_AMR2") && s.CreationTime.Date == new DateTime(2025, 11, 27)) {
//         s.Delete(true);
//         //Console.WriteLine($"Deleted session from {s.CreationTime.Date}");
//     }
// }

In [ ]:
// static var sessions = BoSSSshell.WorkflowMgm.Sessions;
// sessions

## Grid Creation 

In [ ]:
int[] Sizes = new int[] { 3, 4 };
IGridInfo[] Grids = new IGridInfo[Sizes.Length];

double r0 = 0.01; 

for(int i = 0; i < Sizes.Length; i++) {
    int Size = Sizes[i];
    string GridName = $"DropletSpreadingUnderGravity_meshSize{Size}";

    IGridInfo cachedGrid = wmg.Grids.FirstOrDefault(grid => grid.Name == GridName);
    cachedGrid = null;
    if(cachedGrid == null) {
        
        // must create new Grid
        double Lx = (double)Size * r0;
        double[] xNodes = GenericBlas.Linspace(-Lx, Lx, (Size*10) + 1);
        double[] yNodes = GenericBlas.Linspace( 0, 2.0 * r0, 10 + 1);
        var grd = Grid2D.Cartesian2DGrid(xNodes, yNodes);
    
        grd.Name = GridName;
    
        grd.EdgeTagNames.Add(1, "navierslip_linear");
        grd.EdgeTagNames.Add(2, "navierslip_linear_bottom");


        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 0;
            if (Math.Abs(X[1]) <= 1.0e-8)
                et = 2;
            if (Math.Abs(X[1] - (2.0 * r0)) <= 1.0e-8)
                et = 1;
            if (Math.Abs(X[0] + Lx) <= 1.0e-8)
                et = 1;
            if (Math.Abs(X[0] - Lx) <= 1.0e-8)
                et = 1;

            return et;
        });
        
        Grids[i] = wmg.SaveGrid(grd);
        
    } else {
        //Console.WriteLine($"type: {cachedGrid.GetType()}, is IGridInfo? {cachedGrid is IGridInfo}");
        Console.WriteLine("Grid already found in database - identifid by name " + GridName);
        Grids[i] = cachedGrid;
    }
    
}

Opening existing database 'C:\BoSSS-localJobs\CLpaper_DropletSpreadingUnderGravity'.
Grid Edge Tags changed.
An equivalent grid (5341261b-82a5-4e2b-84ae-40830601bb92) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (c62253f6-b37d-46e8-8bf5-4c98558eb46d) is already present in the database -- the grid will not be saved.


## Initial Values

In [10]:
double r = 0.01; 
r

0.01

In [11]:
var PhiFunc = new Formula(
    "Phi",
    false,
    "using ilPSP.Utils; " + 
    "double dropRadius = 0.01;" + 
    "double Phi(double[] X) { " + 
    "    return Math.Sqrt((X[0] - 0.0).Pow2() + (X[1] - 0.0).Pow2()) - dropRadius; " + 
    "}");

## Control settings

In [13]:
string[] EoStudy = { "0.01", "0.1", "1", "2", "5" };

In [14]:
Formula[] GravityValues = new Formula[] {
    new Formula("grav",false,"using ilPSP.Utils; double grav(double[] X) { return -0.0072; }"), // Eo = 0.01
    new Formula("grav",false,"using ilPSP.Utils; double grav(double[] X) { return -0.072; }"),  // Eo = 0.1
    new Formula("grav",false,"using ilPSP.Utils; double grav(double[] X) { return -0.72; }"),   // Eo = 1
    new Formula("grav",false,"using ilPSP.Utils; double grav(double[] X) { return -1.44; }"),   // Eo = 2   
    new Formula("grav",false,"using ilPSP.Utils; double grav(double[] X) { return -3.6; }")     // Eo = 5
};

In [18]:
List<XNSE_Control> Controls = new List<XNSE_Control>();

In [ ]:
for(int iAMR = 1; iAMR <= 2; iAMR++) {
for(int iEo = 0; iEo < EoStudy.Length; iEo++) {

    if (iAMR == 2 && (EoStudy[iEo] == "0.1" || EoStudy[iEo] == "1"))
        continue;
    
    XNSE_Control C = new XNSE_Control();

    C.FieldOptions.Add(VariableNames.GravityY + "#A", new FieldOpts() {
        SaveToDB = FieldOpts.SaveToDBOpt.TRUE
    });
    C.FieldOptions.Add(VariableNames.GravityY + "#B", new FieldOpts() {
        SaveToDB = FieldOpts.SaveToDBOpt.TRUE
    });
    
    int pDeg = 2;   
    var grd  = (EoStudy[iEo] == "5") ? Grids[1] : Grids[0];

    C.SetDGdegree(pDeg);
    
    // set grid
    C.SetGrid(grd);
    
    // initial conditions   
    C.AddInitialValue("Phi", PhiFunc);   

    C.AddInitialValue("GravityY#A", GravityValues[iEo]);
    C.AddInitialValue("GravityY#B", GravityValues[iEo]);
    

    C.PhysicalParameters.rho_A = 1000.0;
    C.PhysicalParameters.rho_B = 1.0;
    C.PhysicalParameters.mu_A  = 1e-2;
    C.PhysicalParameters.mu_B  = 1e-5;
    C.PhysicalParameters.Sigma = 0.072;

    C.PhysicalParameters.IncludeConvection = true;
    C.PhysicalParameters.Material = true;

    C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
    C.PhysicalParameters.betaL = 0.0;
    double hmin = 2.0 * 0.01 / 10.0;
    double beta_scl = 100;
    C.PhysicalParameters.betaS_A = beta_scl * C.PhysicalParameters.mu_A / hmin;
    C.PhysicalParameters.betaS_B = beta_scl * C.PhysicalParameters.mu_B / hmin;
    C.PhysicalParameters.theta_e = Math.PI * (5.0 / 18.0); // 50 degrees
    
    C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;

    C.NonLinearSolver.MaxSolverIterations = 50;
    
    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
    C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
    C.Option_LevelSetEvolution = LevelSetEvolution.FastMarching;
    C.TimeSteppingScheme = TimeSteppingScheme.BDF3;

  
    double dt = (iAMR == 1) ? 1e-4 : 5e-5;
    C.dtMin         = dt;
    C.dtMax         = dt;
    C.Endtime       = 1.0;
    C.NoOfTimesteps = (int) (1.0 / dt); 
    
    C.saveperiod = 100;
    
    C.AdaptiveMeshRefinement = true;
    C.activeAMRlevelIndicators.Add(new AMRonNarrowbandAtBoundary(new byte[] { 2 }) { useUnion = true, maxRefinementLevel = iAMR });
    C.AMR_startUpSweeps = iAMR;
    if (iAMR == 1) {
        C.SessionName = "StaticDropletOnSlipWall_Eo" + EoStudy[iEo];     
    } else {    
        C.FastMarchingReInitPeriod = 200;        
        C.SessionName = "StaticDropletOnSlipWall_Eo" + EoStudy[iEo] + "_AMR2";
    }
    
    Controls.Add(C);
}
}

In [20]:
foreach (var ctrl in Controls) {
    Console.WriteLine($"{ctrl.SessionName}");
    // Console.WriteLine($"{ctrl.SessionName}: dt = {ctrl.dtMin}; saveperiod = {ctrl.saveperiod}; logperiod = {ctrl.PostprocessingModules.Pick(0).LogPeriod}");
}

StaticDropletOnSlipWall_Eo0.01
StaticDropletOnSlipWall_Eo0.1
StaticDropletOnSlipWall_Eo1
StaticDropletOnSlipWall_Eo2
StaticDropletOnSlipWall_Eo5
StaticDropletOnSlipWall_Eo0.01_AMR2
StaticDropletOnSlipWall_Eo2_AMR2
StaticDropletOnSlipWall_Eo5_AMR2


In [17]:
int NoCtrls = Controls.Count();
NoCtrls

3

## Launch/Restart Jobs

In [ ]:
public static Job CreateRestartJob(Job job2rest) {

    string baseName = job2rest.LatestSession.Name;

    // find latest session with same base name (in case of multiple restarts)
    var studySess = sessions.Where(sess => sess.Name.Contains(baseName));
    if (!baseName.Contains("_AMR2")) {
        studySess = sessions.Where(sess => sess.Name.Contains(baseName) && !sess.Name.Contains("_AMR2"));
    }
    int studyCount = studySess.Count();
    studySess.OrderBy(sess => sess.CreationTime);
    var LatestSession = studySess.Last();
    Console.WriteLine($"Creating restart job for session {LatestSession.Name} ...");

    var rCtrl = LatestSession.GetControl().CloneAs();
    // var rCtrl = job2rest.LatestSession.GetControl().CloneAs();

    Guid rst_ID = LatestSession.ID;
    var LastTimestep = LatestSession.Timesteps.Last();
    TimestepNumber rst_ts = LastTimestep.TimeStepNumber;

    rCtrl.GridGuid = LastTimestep.GridID;
    rCtrl.m_Grid = LastTimestep.Grid as IGrid;

    rCtrl.InitialValues.Clear();
    rCtrl.InitialValues_Evaluators.Clear();

    rCtrl.Endtime = 1.0;
    rCtrl.RestartInfo = Tuple.Create(rst_ID, rst_ts);

    rCtrl.SessionName = baseName + "_Restart" + studyCount;

    var rJob = rCtrl.CreateJob();

    // set resources
    rJob.NumberOfMPIProcs = job2rest.NumberOfMPIProcs;

    int numThreads = job2rest.NumberOfThreads;
    rJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = rJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    rJob.MySetCommandLineArguments(args.Values.ToArray());

    rJob.RetryCount = job2rest.RetryCount;

    Console.WriteLine($"Restart job for session {rCtrl.SessionName} created.");

    return rJob;
}

In [ ]:
List<Job> jobs = new List<Job>();

In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    
    oneJob.NumberOfMPIProcs = 1;

    //System.Environment.SetEnvironmentVariable("OMP_NUM_THREADS", "2");
    int numThreads = 1;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

    //oneJob.UseComputeNodesExclusive = true;\n",

    //((SlurmClient)myBatch).ExecutionTime = "24:00:00";

    oneJob.RetryCount = 1;
    oneJob.Activate(myBatch);
    
    if (oneJob.Status == JobStatus.FailedOrCanceled) {
        Console.WriteLine($"Job for session {ctrl.SessionName} failed. Trying to restart latest job ...");

        var restartJob = CreateRestartJob(oneJob);
        restartJob.Activate(myBatch);
        jobs.Add(restartJob);

    } else {   

        //oneJob.Activate(myBatch);
        jobs.Add(oneJob);
    }
}

Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job DropletInEquilibrium_meshStudy_mesh20 ... 
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\XNSEpaper_DropletInEquilibrium-XNSE_Solver2025Oct13_132951.571695
copied 43 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job DropletInEquilibrium_meshStudy_mesh40 ... 
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\XNSEpaper_DropletInEquilibrium-XNSE_Solver2025Oct13_133001.521669
copied 43 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.


## Wait for Completion and Check Job Status

In [ ]:
int NoInProgress = jobs.Where(job => job.Status == JobStatus.InProgress 
                                    || job.Status == JobStatus.PendingInExecutionQueue
                                    || job.Status == JobStatus.PreActivation).Count();
NoInProgress

In [ ]:
int maxDays = 7;
int waitingDays = 0;
while (NoInProgress > 0 && waitingDays < maxDays) {
    wmg.BlockUntilAllJobsTerminate(3600*24*1); // wait one day for the jobs to finish
    NoInProgress = jobs.Where(job => job.Status == JobStatus.InProgress).Count();
    waitingDays++;
}

In [ ]:
int NoFailed = jobs.Where(job => job.Status == JobStatus.FailedOrCanceled).Count();
NoFailed

In [ ]:
int NoSuccess = jobs.Where(job => job.Status == JobStatus.FinishedSuccessful).Count();
NoSuccess

In [ ]:
// check for restart sessions
if (NoFailed > 0) {
    foreach (var ctrl in Controls) {
        var job = ctrl.GetJob();
        if (job.Status == JobStatus.FailedOrCanceled) {
            var studySess = sessions.Where(sess => sess.Name.Contains(ctrl.SessionName));
            int studyCount = studySess.Count();

            if (studyCount > 1) {
                Console.WriteLine("Restart session found! Take last run");       
                bool success = studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single().SuccessfulTermination;
                if (success) {
                    Console.WriteLine($"Restart session {ctrl.SessionName}_restart{studyCount-1} was successful.");
                    NoFailed--;
                    NoSuccess++;
                } else {
                    Console.WriteLine($"Restart session {ctrl.SessionName}_restart{studyCount-1} also failed.");
                }
            } else if (studyCount == 1) {
                Console.WriteLine("No restart session found. {ctrl.SessionName} might to be restarted.");
            } else { // studyCount == 0
                throw new ApplicationException($"No session found for {ctrl.SessionName}!"); // should not happen
            } 
        }
    }

}

In [ ]:
//NUnit.Framework.Assert.AreEqual(0, NoFailed, $"Some simulations failed.");

In [ ]:
//NUnit.Framework.Assert.AreEqual(NoCtrls, NoSuccess, $"Not all simulations finished successfully.");

Error: (1,37): error CS0103: The name 'NoSuccess' does not exist in the current context